## Loan and reconstruct Core

In [8]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

# ------------------------------------------------------------
# File paths
# ------------------------------------------------------------

DATA_DIR = Path(".")   # change if the files are elsewhere

CORE_FILE = DATA_DIR / "atlasnlp_core_v2_all_normalized.csv"
LANG_FILE = DATA_DIR / "automated_full_set_with_audited_languages.csv"


# ------------------------------------------------------------
# Load
# ------------------------------------------------------------

core_all = pd.read_csv(CORE_FILE, low_memory=False)
lang_audit = pd.read_csv(LANG_FILE, low_memory=False)

print("Raw Core rows:", len(core_all))
print("Audited-language rows:", len(lang_audit))


# ------------------------------------------------------------
# Final AtlasNLP-Core population
# ------------------------------------------------------------

core = core_all[
    core_all["v2_primary_eligible"] == True
].copy()

print("\nFinal primary-eligible Core records:", len(core))


# ------------------------------------------------------------
# Bring in the fields needed for the language-evaluation funnel
#
# source_row_index is a one-to-one identifier across these files.
# ------------------------------------------------------------

lang_fields = lang_audit[
    [
        "source_row_index",
        "audited_languages",
        "num_audited_languages",
        "Is dataset in multiple choice format?",
    ]
].copy()

core = core.merge(
    lang_fields,
    on="source_row_index",
    how="left",
    validate="one_to_one",
)

print("Rows after language/MCQ join:", len(core))

assert len(core_all) == 18035
assert len(lang_audit) == 18035
assert len(core) == 13462
assert core["source_row_index"].is_unique

print("\n✓ Final Core reconstructed successfully.")

Raw Core rows: 18035
Audited-language rows: 18035

Final primary-eligible Core records: 13462
Rows after language/MCQ join: 13462

✓ Final Core reconstructed successfully.


## Establish the audited language population

In [9]:
# ------------------------------------------------------------
# Parse final audited languages
# ------------------------------------------------------------

def parse_semicolon_list(x):
    if pd.isna(x) or not str(x).strip():
        return []
    return [
        item.strip()
        for item in str(x).split(";")
        if item.strip()
    ]


core["language_list"] = core["audited_languages"].apply(parse_semicolon_list)

# These were identified during the AtlasNLP language audit as
# programming-language artifacts rather than natural-language coverage.
LANGUAGE_ARTIFACTS = {"Ada", "ABAP"}

core["language_list"] = core["language_list"].apply(
    lambda xs: [x for x in xs if x not in LANGUAGE_ARTIFACTS]
)

core["n_languages"] = core["language_list"].apply(len)
core["has_usable_language"] = core["n_languages"] > 0


# ------------------------------------------------------------
# First funnel pass
# ------------------------------------------------------------

core_lang = core[
    core["has_usable_language"]
].copy()

all_final_languages = sorted({
    language
    for languages in core_lang["language_list"]
    for language in languages
})

print("Final AtlasNLP-Core records:", len(core))
print("Records with >=1 usable audited language:", len(core_lang))
print("Unique audited languages:", len(all_final_languages))

assert len(all_final_languages) == 1239

print("\n✓ Published Core language inventory reproduced.")

Final AtlasNLP-Core records: 13462
Records with >=1 usable audited language: 12723
Unique audited languages: 1239

✓ Published Core language inventory reproduced.


## Restrict to explicit represented-country evidence

In [10]:
# ============================================================
# Cell 2 — Explicit represented-country evidence
# ============================================================

import json


# ------------------------------------------------------------
# Parse the FINAL normalized explicit-country field
# ------------------------------------------------------------

def parse_json_list(x):
    """
    Parse AtlasNLP JSON-list fields such as:
        '["Nigeria", "Ghana"]'
    into Python lists.
    """
    if isinstance(x, list):
        return x

    if pd.isna(x) or not str(x).strip():
        return []

    try:
        value = json.loads(x)
        return value if isinstance(value, list) else []
    except (json.JSONDecodeError, TypeError):
        return []


core["explicit_country_list"] = (
    core["explicit_content_countries_normalized"]
    .apply(parse_json_list)
)

core["n_explicit_countries"] = core["explicit_country_list"].apply(len)


# ------------------------------------------------------------
# Sanity checks against the final AtlasNLP fields
# ------------------------------------------------------------

assert (
    core["n_explicit_countries"]
    == core["n_explicit_countries_normalized"]
).all()

assert (
    (core["n_explicit_countries"] > 0)
    == core["has_explicit_country_normalized"]
).all()

assert (
    core.loc[
        core["has_explicit_country_normalized"],
        "content_country_status_normalized"
    ] == "EXPLICIT"
).all()


# ------------------------------------------------------------
# Published AtlasNLP explicit layer
# ------------------------------------------------------------

n_explicit_all = int(core["has_explicit_country_normalized"].sum())

explicit_assoc_all = int(
    core.loc[
        core["has_explicit_country_normalized"],
        "n_explicit_countries"
    ].sum()
)

print("FINAL ATLASNLP-CORE EXPLICIT LAYER")
print("----------------------------------")
print("Explicit dataset records:", n_explicit_all)
print("Explicit country-record associations:", explicit_assoc_all)


# These should reproduce the published AtlasNLP counts
assert n_explicit_all == 2447
assert explicit_assoc_all == 4421


# ------------------------------------------------------------
# Apply the language requirement from Cell 1
#
# For Language Is Not Enough, a record cannot enter the
# within-language analysis unless it also has a usable
# audited natural-language label.
# ------------------------------------------------------------

core_explicit = core[
    core["has_usable_language"]
    & core["has_explicit_country_normalized"]
].copy()


n_explicit_lang = len(core_explicit)

explicit_assoc_lang = int(
    core_explicit["n_explicit_countries"].sum()
)

explicit_countries_lang = sorted({
    country
    for countries in core_explicit["explicit_country_list"]
    for country in countries
})


print("\nLANGUAGE-EVALUATION POPULATION AFTER EXPLICIT GEOGRAPHY")
print("------------------------------------------------------")
print("Starting final Core records:             ", len(core))
print("With usable audited language:            ", len(core_lang))
print("With usable language + explicit country: ", n_explicit_lang)
print("Country-record associations retained:    ", explicit_assoc_lang)
print("Unique represented countries retained:   ", len(explicit_countries_lang))


# ------------------------------------------------------------
# How much did requiring language remove from the explicit layer?
# ------------------------------------------------------------

print("\nExplicit records excluded for no usable language:",
      n_explicit_all - n_explicit_lang)

print("Explicit country associations excluded:",
      explicit_assoc_all - explicit_assoc_lang)

FINAL ATLASNLP-CORE EXPLICIT LAYER
----------------------------------
Explicit dataset records: 2447
Explicit country-record associations: 4421

LANGUAGE-EVALUATION POPULATION AFTER EXPLICIT GEOGRAPHY
------------------------------------------------------
Starting final Core records:              13462
With usable audited language:             12723
With usable language + explicit country:  2429
Country-record associations retained:     4369
Unique represented countries retained:    158

Explicit records excluded for no usable language: 18
Explicit country associations excluded: 52


## Keep only unambiguous language–country pairings

In [11]:
# ============================================================
# Cell 3 — Controlled language-country pairing
# ============================================================

# ------------------------------------------------------------
# A record is safely pairable when:
#
#   A. exactly 1 language + >=1 explicit countries
#      e.g. Arabic -> Egypt, Jordan, Palestine
#
#   OR
#
#   B. >1 languages + exactly 1 explicit country
#      e.g. Hausa, Yoruba, Igbo -> Nigeria
#
# We exclude:
#   >1 languages + >1 countries
#
# because record-level metadata does not establish which
# language corresponds to which country.
# ------------------------------------------------------------

core_explicit["controlled_pairing"] = (
    (
        (core_explicit["n_languages"] == 1)
        & (core_explicit["n_explicit_countries"] >= 1)
    )
    |
    (
        (core_explicit["n_languages"] > 1)
        & (core_explicit["n_explicit_countries"] == 1)
    )
)


core_controlled = core_explicit[
    core_explicit["controlled_pairing"]
].copy()

core_ambiguous = core_explicit[
    ~core_explicit["controlled_pairing"]
].copy()


# ------------------------------------------------------------
# Breakdown by the two ways a record can qualify
# ------------------------------------------------------------

mono_language = (
    (core_controlled["n_languages"] == 1)
    & (core_controlled["n_explicit_countries"] >= 1)
).sum()

multi_language_one_country = (
    (core_controlled["n_languages"] > 1)
    & (core_controlled["n_explicit_countries"] == 1)
).sum()


print("CONTROLLED LANGUAGE-COUNTRY PAIRING")
print("-----------------------------------")
print("Language + explicit-country records:", len(core_explicit))
print("Controlled records retained:        ", len(core_controlled))
print("Ambiguous multi-language/multi-country records excluded:",
      len(core_ambiguous))

print("\nControlled records by route:")
print("  1 language + >=1 countries:       ", mono_language)
print("  >1 languages + exactly 1 country: ", multi_language_one_country)

assert len(core_controlled) == mono_language + multi_language_one_country

CONTROLLED LANGUAGE-COUNTRY PAIRING
-----------------------------------
Language + explicit-country records: 2429
Controlled records retained:         2231
Ambiguous multi-language/multi-country records excluded: 198

Controlled records by route:
  1 language + >=1 countries:        1753
  >1 languages + exactly 1 country:  478


## Expand only the defensible pairings

In [13]:
# ============================================================
# Cell 3b — Expand controlled records into language-country rows
# ============================================================

controlled_pairs = (
    core_controlled
    .explode("language_list")
    .explode("explicit_country_list")
    .rename(columns={
        "language_list": "language",
        "explicit_country_list": "country",
    })
    .copy()
)

# Defensive cleanup
controlled_pairs = controlled_pairs[
    controlled_pairs["language"].notna()
    & controlled_pairs["country"].notna()
].copy()


print("CONTROLLED EXPANDED POPULATION")
print("------------------------------")
print("Underlying dataset records:", controlled_pairs["source_row_index"].nunique())
print("Language-country rows:", len(controlled_pairs))
print("Unique languages:", controlled_pairs["language"].nunique())
print("Unique represented countries:", controlled_pairs["country"].nunique())

assert controlled_pairs["source_row_index"].nunique() == 2231

CONTROLLED EXPANDED POPULATION
------------------------------
Underlying dataset records: 2231
Language-country rows: 4743
Unique languages: 435
Unique represented countries: 147


## Evaluation-eligible records

In [15]:
# ============================================================
# Cell 4 — Evaluation-task eligibility
# ============================================================

QA_TASK = "Question Answering"
NLI_TASK = "Semantics: Lexical and Sentence Level, Textual Inference"


# ------------------------------------------------------------
# Reproduce the old funnel criterion:
#
#   Question Answering
#       OR
#   NLI / Textual Inference
#       OR
#   explicitly multiple-choice
#
# MCQ is allowed regardless of the normalized task category,
# since an MCQ benchmark in another task can still support
# controlled model evaluation.
# ------------------------------------------------------------

is_qa = (
    core_controlled["normalized_task_category"]
    .eq(QA_TASK)
)

is_nli = (
    core_controlled["normalized_task_category"]
    .eq(NLI_TASK)
)

is_mcq = (
    core_controlled["Is dataset in multiple choice format?"]
    .astype(str)
    .str.strip()
    .str.casefold()
    .eq("yes")
)

core_controlled["evaluation_eligible"] = (
    is_qa | is_nli | is_mcq
)

core_eval = core_controlled[
    core_controlled["evaluation_eligible"]
].copy()


# ------------------------------------------------------------
# Report the pass
# ------------------------------------------------------------

print("EVALUATION ELIGIBILITY")
print("----------------------")
print("Controlled records entering pass:", len(core_controlled))
print("Evaluation-eligible records:      ", len(core_eval))
print()

print("Individual criteria:")
print("  Question Answering:", int(is_qa.sum()))
print("  NLI / Textual Inference:", int(is_nli.sum()))
print("  Multiple choice:", int(is_mcq.sum()))

EVALUATION ELIGIBILITY
----------------------
Controlled records entering pass: 2231
Evaluation-eligible records:       297

Individual criteria:
  Question Answering: 118
  NLI / Textual Inference: 6
  Multiple choice: 221


### eligibility routes overlap

In [16]:
# ============================================================
# Cell 4b — Eligibility-route overlap
# ============================================================

eligibility_routes = pd.DataFrame({
    "QA": is_qa,
    "NLI": is_nli,
    "MCQ": is_mcq,
})

route_counts = (
    eligibility_routes
    .value_counts()
    .reset_index(name="records")
    .sort_values("records", ascending=False)
)

print(route_counts.to_string(index=False))

   QA   NLI   MCQ  records
False False False     1934
False False  True      173
 True False False       70
 True False  True       48
False  True False        6


## expand the eligible records safely

In [17]:
# ============================================================
# Cell 4c — Expand evaluation-eligible records
# ============================================================

eval_pairs = (
    core_eval
    .explode("language_list")
    .explode("explicit_country_list")
    .rename(columns={
        "language_list": "language",
        "explicit_country_list": "country",
    })
    .copy()
)

eval_pairs = eval_pairs[
    eval_pairs["language"].notna()
    & eval_pairs["country"].notna()
].copy()


print("EVALUATION-ELIGIBLE EXPANDED POPULATION")
print("---------------------------------------")
print("Underlying dataset records:", eval_pairs["source_row_index"].nunique())
print("Language-country rows:", len(eval_pairs))
print("Unique languages:", eval_pairs["language"].nunique())
print("Unique represented countries:", eval_pairs["country"].nunique())

assert eval_pairs["source_row_index"].nunique() == len(core_eval)

EVALUATION-ELIGIBLE EXPANDED POPULATION
---------------------------------------
Underlying dataset records: 297
Language-country rows: 616
Unique languages: 81
Unique represented countries: 84


## Languages with ≥2 and ≥3 countries

In [18]:
# ============================================================
# Cell 5 — Cross-country coverage by language
# ============================================================

# ------------------------------------------------------------
# Count DISTINCT explicitly represented countries per language
# across the evaluation-eligible controlled population.
# ------------------------------------------------------------

language_country_counts = (
    eval_pairs
    .groupby("language")["country"]
    .nunique()
    .sort_values(ascending=False)
)

languages_ge2 = language_country_counts[
    language_country_counts >= 2
]

languages_ge3 = language_country_counts[
    language_country_counts >= 3
]


print("CROSS-COUNTRY LANGUAGE COVERAGE")
print("-------------------------------")
print("Evaluation-eligible languages:", len(language_country_counts))
print("Languages with >=2 countries: ", len(languages_ge2))
print("Languages with >=3 countries: ", len(languages_ge3))


# ------------------------------------------------------------
# Show the actual qualifying languages
# ------------------------------------------------------------

language_coverage = (
    language_country_counts
    .rename("n_countries")
    .reset_index()
)

language_coverage["ge_2_countries"] = (
    language_coverage["n_countries"] >= 2
)

language_coverage["ge_3_countries"] = (
    language_coverage["n_countries"] >= 3
)

print("\nLanguages spanning >=2 countries:")
print(
    language_coverage[
        language_coverage["ge_2_countries"]
    ].to_string(index=False)
)

CROSS-COUNTRY LANGUAGE COVERAGE
-------------------------------
Evaluation-eligible languages: 81
Languages with >=2 countries:  32
Languages with >=3 countries:  18

Languages spanning >=2 countries:
  language  n_countries  ge_2_countries  ge_3_countries
   English           53            True            True
    Arabic           22            True            True
    French           13            True            True
   Italian            7            True            True
   Spanish            5            True            True
    German            5            True            True
   Russian            4            True            True
   Chinese            4            True            True
     Greek            3            True            True
  Japanese            3            True            True
Indonesian            3            True            True
    Korean            3            True            True
     Hindi            3            True            True
Portuguese     

## How much of the 297-record population belongs to those languages?

In [20]:
# ============================================================
# Cell 5b — Records belonging to multi-country language ecosystems
# ============================================================

langs_ge2 = set(languages_ge2.index)
langs_ge3 = set(languages_ge3.index)

eval_pairs_ge2 = eval_pairs[
    eval_pairs["language"].isin(langs_ge2)
].copy()

eval_pairs_ge3 = eval_pairs[
    eval_pairs["language"].isin(langs_ge3)
].copy()


print("MULTI-COUNTRY LANGUAGE ECOSYSTEMS")
print("---------------------------------")

print("Languages with >=2 countries:",
      len(langs_ge2))

print("Underlying records in those language ecosystems:",
      eval_pairs_ge2["source_row_index"].nunique())

print()

print("Languages with >=3 countries:",
      len(langs_ge3))

print("Underlying records in those language ecosystems:",
      eval_pairs_ge3["source_row_index"].nunique())

MULTI-COUNTRY LANGUAGE ECOSYSTEMS
---------------------------------
Languages with >=2 countries: 32
Underlying records in those language ecosystems: 286

Languages with >=3 countries: 18
Underlying records in those language ecosystems: 259


## Strict explicit coordinated candidates

In [21]:
# ============================================================
# Cell 6A — Strict coordinated multi-country candidates
#            (EXPLICIT geography only)
# ============================================================

# Because core_eval already satisfies the controlled-pairing rule,
# any record with >=2 countries necessarily has exactly one
# usable language. We can therefore safely interpret its countries
# as belonging to that language.

core_coordinated_explicit = core_eval[
    core_eval["n_explicit_countries"] >= 2
].copy()

core_coordinated_explicit["language"] = (
    core_coordinated_explicit["language_list"].str[0]
)


print("STRICT COORDINATED MULTI-COUNTRY CANDIDATES")
print("-------------------------------------------")
print("Evaluation-eligible records:", len(core_eval))
print("Records individually spanning >=2 explicit countries:",
      len(core_coordinated_explicit))

print("Records spanning >=3 explicit countries:",
      int((core_coordinated_explicit["n_explicit_countries"] >= 3).sum()))

print("Languages represented:",
      core_coordinated_explicit["language"].nunique())

print("\nCandidates by language:")
print(
    core_coordinated_explicit["language"]
    .value_counts()
    .to_string()
)

STRICT COORDINATED MULTI-COUNTRY CANDIDATES
-------------------------------------------
Evaluation-eligible records: 297
Records individually spanning >=2 explicit countries: 37
Records spanning >=3 explicit countries: 23
Languages represented: 3

Candidates by language:
English    32
Arabic      3
French      2


### Recall-oriented candidate retrieval

In [22]:
# ============================================================
# Cell 6B — Recall-oriented candidate retrieval
#            (EXPLICIT + INFERRED geography)
# ============================================================

# ------------------------------------------------------------
# Parse final explicit+inferred geography
# ------------------------------------------------------------

core["any_country_list"] = (
    core["explicit_plus_inferred_countries_normalized"]
    .apply(parse_json_list)
)

core["n_any_countries"] = core["any_country_list"].apply(len)


# ------------------------------------------------------------
# Require usable language + some final geographic evidence
# ------------------------------------------------------------

core_any_geo = core[
    core["has_usable_language"]
    & core["has_any_country_normalized"]
].copy()


# ------------------------------------------------------------
# Reapply the SAME controlled-pairing principle
# using the broader geography layer
# ------------------------------------------------------------

core_any_geo["controlled_pairing_any"] = (
    (
        (core_any_geo["n_languages"] == 1)
        & (core_any_geo["n_any_countries"] >= 1)
    )
    |
    (
        (core_any_geo["n_languages"] > 1)
        & (core_any_geo["n_any_countries"] == 1)
    )
)

core_any_controlled = core_any_geo[
    core_any_geo["controlled_pairing_any"]
].copy()


# ------------------------------------------------------------
# Same evaluation-eligibility rule as before
# ------------------------------------------------------------

is_qa_any = (
    core_any_controlled["normalized_task_category"]
    .eq(QA_TASK)
)

is_nli_any = (
    core_any_controlled["normalized_task_category"]
    .eq(NLI_TASK)
)

is_mcq_any = (
    core_any_controlled["Is dataset in multiple choice format?"]
    .astype(str)
    .str.strip()
    .str.casefold()
    .eq("yes")
)

core_any_controlled["evaluation_eligible"] = (
    is_qa_any | is_nli_any | is_mcq_any
)

core_eval_any = core_any_controlled[
    core_any_controlled["evaluation_eligible"]
].copy()


# ------------------------------------------------------------
# Coordinated candidate = one record itself spans >=2 countries
# ------------------------------------------------------------

core_coordinated_any = core_eval_any[
    core_eval_any["n_any_countries"] >= 2
].copy()

core_coordinated_any["language"] = (
    core_coordinated_any["language_list"].str[0]
)


print("RECALL-ORIENTED CANDIDATE RETRIEVAL")
print("-----------------------------------")
print("Usable language + any geography:", len(core_any_geo))
print("Controlled pairing:", len(core_any_controlled))
print("Evaluation eligible:", len(core_eval_any))
print("Coordinated >=2-country candidates:", len(core_coordinated_any))

print("\nFinal geography status among candidates:")
print(
    core_coordinated_any[
        "content_country_status_normalized"
    ].value_counts().to_string()
)

RECALL-ORIENTED CANDIDATE RETRIEVAL
-----------------------------------
Usable language + any geography: 3483
Controlled pairing: 3187
Evaluation eligible: 376
Coordinated >=2-country candidates: 43

Final geography status among candidates:
EXPLICIT         38
INFERRED_ONLY     5


In [23]:
# ============================================================
# Cell 6C — Candidates recovered only by broader retrieval
# ============================================================

strict_ids = set(
    core_coordinated_explicit["source_row_index"]
)

recovered_candidates = core_coordinated_any[
    ~core_coordinated_any["source_row_index"].isin(strict_ids)
].copy()


show_cols = [
    "source_row_index",
    "Dataset name",
    "language",
    "content_country_status_normalized",
    "explicit_country_list",
    "any_country_list",
    "normalized_task_category",
    "Is dataset in multiple choice format?",
]

print("ADDITIONAL CANDIDATES RECOVERED")
print("--------------------------------")
print("N =", len(recovered_candidates))
print()

print(
    recovered_candidates[show_cols]
    .to_string(index=False)
)

ADDITIONAL CANDIDATES RECOVERED
--------------------------------
N = 6

 source_row_index                        Dataset name language content_country_status_normalized explicit_country_list                                             any_country_list                                 normalized_task_category Is dataset in multiple choice format?
              183                               M-MD3  English                     INFERRED_ONLY                    []                                       [United States, India]                         Dialogue and Interactive Systems                                   Yes
              988                               FRIDA  English                     INFERRED_ONLY                    []                                              [Turkey, Syria]                              Natural Language Generation                                   Yes
             3155                          CryptOpiQA  English                          EXPLICIT       

## Reconcile the 43 candidates with the old manual audit

In [25]:
# ============================================================
# Cell 7 — Reconcile new candidates with the old manual audit
# ============================================================

from pathlib import Path
import pandas as pd
import re


FUNNEL_FILE = DATA_DIR / "Funnel.xlsx"


# ------------------------------------------------------------
# Load the old 134-candidate manual audit
# ------------------------------------------------------------

old_audit = pd.read_excel(
    FUNNEL_FILE,
    sheet_name="cross-country candidates"
)

print("Old manually audited candidates:", len(old_audit))


# ------------------------------------------------------------
# Normalization helpers
# ------------------------------------------------------------

def normalize_url(x):
    if pd.isna(x):
        return ""

    x = str(x).strip().lower()

    # remove query / fragment
    x = x.split("?")[0].split("#")[0]

    # ACL links sometimes differ only by .pdf or trailing slash
    if x.endswith(".pdf"):
        x = x[:-4]

    return x.rstrip("/")


def normalize_dataset_name(x):
    if pd.isna(x):
        return ""

    x = str(x).strip().lower()
    x = re.sub(r"\s+", " ", x)

    return x


def parse_old_country_string(x):
    if pd.isna(x) or not str(x).strip():
        return []

    return [
        c.strip()
        for c in str(x).split(";")
        if c.strip()
    ]


# ------------------------------------------------------------
# Prepare OLD audit identity fields
# ------------------------------------------------------------

old_audit["audit_paper_key"] = (
    old_audit["Paper Link (URL)"]
    .apply(normalize_url)
)

old_audit["audit_dataset_key"] = (
    old_audit["Dataset name"]
    .apply(normalize_dataset_name)
)

old_audit["identity_key"] = (
    old_audit["audit_paper_key"]
    + "||"
    + old_audit["audit_dataset_key"]
)

old_audit["old_country_list"] = (
    old_audit["countries"]
    .apply(parse_old_country_string)
)


# ------------------------------------------------------------
# Prepare the NEW 43-candidate population
# ------------------------------------------------------------

candidate_audit = core_coordinated_any.copy()

candidate_audit["language"] = (
    candidate_audit["language_list"].str[0]
)

candidate_audit["current_paper_key"] = (
    candidate_audit["paper_url"]
    .apply(normalize_url)
)

candidate_audit["current_dataset_key"] = (
    candidate_audit["Dataset name"]
    .apply(normalize_dataset_name)
)

candidate_audit["identity_key"] = (
    candidate_audit["current_paper_key"]
    + "||"
    + candidate_audit["current_dataset_key"]
)


# ------------------------------------------------------------
# Keep useful prior-audit columns
# ------------------------------------------------------------

prior_cols = [
    "identity_key",
    "language",
    "Dataset name",
    "Paper Link (URL)",
    "countries",
    "old_country_list",
    "Country_split verified?",
    "Per country counts as tuples",
    "Minimum country count",
    "Split unit",
    "Same task and label format? ",
    "Any confounders e.g. level, topic etc",
    "Verdict? Pass or fail",
    "Evidence source",
    "Verifiable? If no, notes",
]

prior = old_audit[prior_cols].copy()

prior = prior.rename(columns={
    "language": "prior_language",
    "Dataset name": "prior_dataset_name",
    "Paper Link (URL)": "prior_paper_url",
    "countries": "prior_countries",
    "Country_split verified?": "prior_country_split_verified",
    "Per country counts as tuples": "prior_per_country_counts",
    "Minimum country count": "prior_minimum_country_n",
    "Split unit": "prior_split_unit",
    "Same task and label format? ": "prior_same_task_and_labels",
    "Any confounders e.g. level, topic etc": "prior_confounds",
    "Verdict? Pass or fail": "prior_verdict",
    "Evidence source": "prior_evidence",
    "Verifiable? If no, notes": "prior_verification_notes",
})


# ------------------------------------------------------------
# Exact paper + dataset match
# ------------------------------------------------------------

candidate_audit = candidate_audit.merge(
    prior,
    on="identity_key",
    how="left",
    validate="one_to_one",
)

candidate_audit["has_exact_prior_audit"] = (
    candidate_audit["prior_verdict"].notna()
)


print("NEW CANDIDATE RECONCILIATION")
print("----------------------------")
print("New Core candidates:", len(candidate_audit))
print(
    "Exact prior paper+dataset audits:",
    int(candidate_audit["has_exact_prior_audit"].sum())
)
print(
    "No exact prior audit:",
    int((~candidate_audit["has_exact_prior_audit"]).sum())
)

Old manually audited candidates: 134
NEW CANDIDATE RECONCILIATION
----------------------------
New Core candidates: 43
Exact prior paper+dataset audits: 30
No exact prior audit: 13


### Check whether final AtlasNLP geography changed

In [26]:
# ============================================================
# Cell 7b — Compare old audited candidate geography
#            with final AtlasNLP geography
# ============================================================

def country_set_equal(old_list, new_list):
    if not isinstance(old_list, list):
        return False
    if not isinstance(new_list, list):
        return False

    return set(old_list) == set(new_list)


exact_prior = candidate_audit[
    candidate_audit["has_exact_prior_audit"]
].copy()

exact_prior["same_country_set"] = exact_prior.apply(
    lambda row: country_set_equal(
        row["old_country_list"],
        row["any_country_list"]
    ),
    axis=1
)


print("EXACT PRIOR AUDITS")
print("------------------")
print("Exact paper+dataset matches:", len(exact_prior))
print(
    "Same AtlasNLP country set:",
    int(exact_prior["same_country_set"].sum())
)
print(
    "Country set changed:",
    int((~exact_prior["same_country_set"]).sum())
)

EXACT PRIOR AUDITS
------------------
Exact paper+dataset matches: 30
Same AtlasNLP country set: 19
Country set changed: 11


### Find the genuinely new/reconciliation cases

In [27]:
# ============================================================
# Cell 7c — Classify cases requiring attention
# ============================================================

# Dataset-name lookup from the old audit, regardless of paper
old_names = set(
    old_audit["audit_dataset_key"]
)

candidate_audit["same_name_seen_in_old_audit"] = (
    candidate_audit["current_dataset_key"]
    .isin(old_names)
)

candidate_audit["review_status"] = np.select(
    [
        candidate_audit["has_exact_prior_audit"],
        (
            ~candidate_audit["has_exact_prior_audit"]
            & candidate_audit["same_name_seen_in_old_audit"]
        ),
    ],
    [
        "EXACT PRIOR AUDIT",
        "SAME DATASET NAME / DIFFERENT PAPER",
    ],
    default="NEW CANDIDATE",
)


print(
    candidate_audit["review_status"]
    .value_counts()
    .to_string()
)

EXACT PRIOR AUDIT                      30
NEW CANDIDATE                          11
SAME DATASET NAME / DIFFERENT PAPER     2


In [28]:
needs_attention = candidate_audit[
    candidate_audit["review_status"] != "EXACT PRIOR AUDIT"
].copy()

attention_cols = [
    "Dataset name",
    "language",
    "paper_url",
    "content_country_status_normalized",
    "any_country_list",
    "normalized_task_category",
    "Is dataset in multiple choice format?",
    "review_status",
]

print(
    needs_attention[attention_cols]
    .sort_values(["language", "Dataset name"])
    .to_string(index=False)
)

                  Dataset name language                                      paper_url content_country_status_normalized                                                                                                               any_country_list                                              normalized_task_category Is dataset in multiple choice format?                       review_status
                   ArabCulture   Arabic     https://aclanthology.org/2025.acl-long.380                          EXPLICIT [Saudi Arabia, Yemen, United Arab Emirates, Syria, Jordan, Palestine, Lebanon, Morocco, Algeria, Tunisia, Libya, Egypt, Sudan] Computational Social Science, Cultural Analytics, NLP for social good                                   Yes SAME DATASET NAME / DIFFERENT PAPER
                   SemRel-2024   Arabic    https://aclanthology.org/2024.semeval-1.128                     INFERRED_ONLY                                                                                                

## Recovery check

In [30]:
# ============================================================
# Cell 7d — Recovery check for previously verified resources
# ============================================================

passed_old = pd.read_excel(
    FUNNEL_FILE,
    sheet_name="Passed"
).copy()

passed_old["dataset_key"] = (
    passed_old["Dataset name"]
    .apply(normalize_dataset_name)
)

new_candidate_names = set(
    candidate_audit["current_dataset_key"]
)

passed_old["retrieved_by_final_funnel"] = (
    passed_old["dataset_key"].isin(new_candidate_names)
)

not_retrieved = passed_old[
    ~passed_old["retrieved_by_final_funnel"]
].copy()

print("Previously verified resources:", len(passed_old))
print(
    "Retrieved by final AtlasNLP funnel:",
    int(passed_old["retrieved_by_final_funnel"].sum())
)
print(
    "Not retrieved:",
    len(not_retrieved)
)

print("\nPreviously verified resources not retrieved:")
print(
    not_retrieved[
        [
            "Dataset name",
            "language",
            "countries",
            "Paper Link (URL)",
            "Country_split verified?",
            "Verdict? Pass or fail",
        ]
    ].to_string(index=False)
)

Previously verified resources: 13
Retrieved by final AtlasNLP funnel: 11
Not retrieved: 2

Previously verified resources not retrieved:
                                    Dataset name language                                           countries                         Paper Link (URL) Country_split verified? Verdict? Pass or fail
Gold Standard Dataset for Racist Tweet Detection  English Canada; India; Japan; United Kingdom; United States       https://aclanthology.org/I17-1093/                     YES  USABLE WITH CONTROLS
       LanguageARC Ethnicity Identification Task  English                       United Kingdom; United States https://aclanthology.org/2020.cllrd-1.7/                     YES  USABLE WITH CONTROLS


In [31]:
# ============================================================
# Cell 8 — Export final candidate set for manual verification
#
# 43 candidates retrieved by the final explicit+inferred funnel
# + 2 previously verified manual-recovery candidates:
#   - Gold Standard Dataset for Racist Tweet Detection
#   - LanguageARC Ethnicity Identification Task
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json


OUTPUT_FILE = Path("language_not_enough_core_cross_country_candidates.csv")


# ------------------------------------------------------------
# Helper
# ------------------------------------------------------------

def safe_parse_json_list(x):
    if isinstance(x, list):
        return x

    if pd.isna(x) or not str(x).strip():
        return []

    try:
        value = json.loads(x)
        return value if isinstance(value, list) else []
    except (json.JSONDecodeError, TypeError):
        return []


def list_to_text(x):
    if isinstance(x, list):
        return "; ".join(map(str, x))
    return ""


# ============================================================
# 1. Start with the 43 candidates retrieved by the
#    explicit+inferred recall-oriented funnel
# ============================================================

export_candidates = core_coordinated_any.copy()


# Strict explicit candidates from Cell 6A
strict_explicit_ids = set(
    core_coordinated_explicit["source_row_index"]
)


export_candidates["candidate_type"] = np.where(
    export_candidates["source_row_index"].isin(strict_explicit_ids),
    "STRICT EXPLICIT CANDIDATE",
    "RECOVERED BY EXPLICIT+INFERRED RETRIEVAL"
)

export_candidates["strict_explicit_candidate"] = np.where(
    export_candidates["source_row_index"].isin(strict_explicit_ids),
    "Yes",
    "No"
)


# Parse inferred geography separately for transparency
export_candidates["inferred_country_list"] = (
    export_candidates["inferred_content_countries_normalized"]
    .apply(safe_parse_json_list)
)


# Human-readable country fields
export_candidates["explicit_countries"] = (
    export_candidates["explicit_country_list"]
    .apply(list_to_text)
)

export_candidates["inferred_countries"] = (
    export_candidates["inferred_country_list"]
    .apply(list_to_text)
)

export_candidates["candidate_countries"] = (
    export_candidates["any_country_list"]
    .apply(list_to_text)
)

export_candidates["n_candidate_countries"] = (
    export_candidates["any_country_list"]
    .apply(len)
)


# No missed reason for algorithmically retrieved candidates
export_candidates["missed_reason"] = ""
export_candidates["manual_recovery_note"] = ""


# ============================================================
# 2. Recover the two previously verified resources that
#    dropped out of the final automated funnel
# ============================================================

RECOVERY_INFO = {
    
    "Gold Standard Dataset for Racist Tweet Detection": {
        
        "candidate_countries": (
            "United States; United Kingdom; India; Japan; Canada"
        ),
        
        "missed_reason": (
            "Missed before candidate retrieval because the final "
            "post-audit AtlasNLP pipeline classified the record as "
            "ineligible / target dataset not confidently recovered "
            "(EXCLUDED_TARGET_NOT_FOUND)."
        ),
        
        "manual_recovery_note": (
            "Source-level inspection verifies a country-separated "
            "racist-tweet classification dataset with per-country "
            "sample counts. Retained as a manual recovery candidate."
        ),
    },

    "LanguageARC Ethnicity Identification Task": {
        
        "candidate_countries": (
            "United Kingdom; United States"
        ),
        
        "missed_reason": (
            "Missed at the >=2-country retrieval stage because final "
            "AtlasNLP geography retains only the United Kingdom, leaving "
            "the record with one represented country."
        ),
        
        "manual_recovery_note": (
            "Source-level inspection verifies participants from both "
            "the United Kingdom and United States and reports "
            "country-separated judgments. Retained as a geography "
            "under-attribution recovery candidate."
        ),
    },
}


# ------------------------------------------------------------
# We need the original 18,035-row Core here because the racist
# tweet record was removed from final eligible Core.
# ------------------------------------------------------------

recovery_rows = []

for dataset_name, info in RECOVERY_INFO.items():

    match = core_all[
        core_all["Dataset name"].eq(dataset_name)
    ].copy()

    if len(match) != 1:
        raise ValueError(
            f"Expected exactly one AtlasNLP record for "
            f"{dataset_name!r}; found {len(match)}."
        )

    row = match.iloc[0].copy()

    # Find its audited-language / MCQ metadata
    source_id = row["source_row_index"]

    lang_match = lang_audit[
        lang_audit["source_row_index"].eq(source_id)
    ]

    if len(lang_match) != 1:
        raise ValueError(
            f"Could not uniquely match audited language information "
            f"for source_row_index={source_id}"
        )

    lang_row = lang_match.iloc[0]


    # Parse language
    language_list = parse_semicolon_list(
        lang_row["audited_languages"]
    )

    language_list = [
        x for x in language_list
        if x not in LANGUAGE_ARTIFACTS
    ]


    # Parse final AtlasNLP geography
    explicit_list = safe_parse_json_list(
        row["explicit_content_countries_normalized"]
    )

    inferred_list = safe_parse_json_list(
        row["inferred_content_countries_normalized"]
    )

    any_list = safe_parse_json_list(
        row["explicit_plus_inferred_countries_normalized"]
    )


    recovery_rows.append({

        "source_row_index": source_id,

        "Dataset name": dataset_name,

        "language": (
            language_list[0]
            if len(language_list) == 1
            else "; ".join(language_list)
        ),

        "paper_url": row["paper_url"],

        "Dataset Link (URL)": row["Dataset Link (URL)"],

        "normalized_task_category":
            row["normalized_task_category"],

        "Is dataset in multiple choice format?":
            lang_row["Is dataset in multiple choice format?"],

        "content_country_status_normalized":
            row["content_country_status_normalized"],

        "explicit_countries":
            list_to_text(explicit_list),

        "inferred_countries":
            list_to_text(inferred_list),

        # Use source-level verified geography here
        "candidate_countries":
            info["candidate_countries"],

        "n_candidate_countries":
            len(info["candidate_countries"].split("; ")),

        "strict_explicit_candidate":
            "No",

        "candidate_type":
            "MANUAL RECOVERY CANDIDATE",

        "v2_primary_eligible":
            row["v2_primary_eligible"],

        "v2_dataset_match_status":
            row["v2_dataset_match_status"],

        "v2_dataset_relationship":
            row["v2_dataset_relationship"],

        "missed_reason":
            info["missed_reason"],

        "manual_recovery_note":
            info["manual_recovery_note"],
    })


recovery_df = pd.DataFrame(recovery_rows)


# ============================================================
# 3. Select and standardize columns for the 43 retrieved rows
# ============================================================

retrieved_export = pd.DataFrame({

    "source_row_index":
        export_candidates["source_row_index"],

    "Dataset name":
        export_candidates["Dataset name"],

    "language":
        export_candidates["language"],

    "paper_url":
        export_candidates["paper_url"],

    "Dataset Link (URL)":
        export_candidates["Dataset Link (URL)"],

    "normalized_task_category":
        export_candidates["normalized_task_category"],

    "Is dataset in multiple choice format?":
        export_candidates["Is dataset in multiple choice format?"],

    "content_country_status_normalized":
        export_candidates["content_country_status_normalized"],

    "explicit_countries":
        export_candidates["explicit_countries"],

    "inferred_countries":
        export_candidates["inferred_countries"],

    "candidate_countries":
        export_candidates["candidate_countries"],

    "n_candidate_countries":
        export_candidates["n_candidate_countries"],

    "strict_explicit_candidate":
        export_candidates["strict_explicit_candidate"],

    "candidate_type":
        export_candidates["candidate_type"],

    "v2_primary_eligible":
        export_candidates["v2_primary_eligible"],

    "v2_dataset_match_status":
        export_candidates["v2_dataset_match_status"],

    "v2_dataset_relationship":
        export_candidates["v2_dataset_relationship"],

    "missed_reason":
        export_candidates["missed_reason"],

    "manual_recovery_note":
        export_candidates["manual_recovery_note"],
})


# ============================================================
# 4. Combine
#
# IMPORTANT:
# The 43 automatically retrieved candidates remain first.
# The two manual recoveries are deliberately appended last.
# ============================================================

final_candidates = pd.concat(
    [
        retrieved_export,
        recovery_df,
    ],
    ignore_index=True
)

final_candidates.insert(
    0,
    "candidate_order",
    range(1, len(final_candidates) + 1)
)


# ============================================================
# 5. Sanity checks
# ============================================================

assert len(retrieved_export) == 43
assert len(recovery_df) == 2
assert len(final_candidates) == 45

assert list(final_candidates.tail(2)["Dataset name"]) == [
    "Gold Standard Dataset for Racist Tweet Detection",
    "LanguageARC Ethnicity Identification Task",
]


print("FINAL CANDIDATE SET")
print("-------------------")
print("Automatically retrieved:", len(retrieved_export))
print("Manual recoveries:", len(recovery_df))
print("Total:", len(final_candidates))

print("\nCandidate types:")
print(
    final_candidates["candidate_type"]
    .value_counts()
    .to_string()
)

print("\nLast two rows:")
print(
    final_candidates.tail(2)[
        [
            "candidate_order",
            "Dataset name",
            "candidate_type",
            "missed_reason",
        ]
    ].to_string(index=False)
)


# ============================================================
# 6. Save
# ============================================================

final_candidates.to_csv(
    OUTPUT_FILE,
    index=False
)

print(f"\nSaved to: {OUTPUT_FILE.resolve()}")

FINAL CANDIDATE SET
-------------------
Automatically retrieved: 43
Manual recoveries: 2
Total: 45

Candidate types:
STRICT EXPLICIT CANDIDATE                   37
RECOVERED BY EXPLICIT+INFERRED RETRIEVAL     6
MANUAL RECOVERY CANDIDATE                    2

Last two rows:
 candidate_order                                     Dataset name            candidate_type                                                                                                                                                                                missed_reason
              44 Gold Standard Dataset for Racist Tweet Detection MANUAL RECOVERY CANDIDATE Missed before candidate retrieval because the final post-audit AtlasNLP pipeline classified the record as ineligible / target dataset not confidently recovered (EXCLUDED_TARGET_NOT_FOUND).
              45        LanguageARC Ethnicity Identification Task MANUAL RECOVERY CANDIDATE                                 Missed at the >=2-country retrieval stag

In [32]:
# ============================================================
# 5b. Attach prior exact-audit information
# ============================================================

# Build exact identity key for final candidate export
final_candidates["current_paper_key"] = (
    final_candidates["paper_url"]
    .apply(normalize_url)
)

final_candidates["current_dataset_key"] = (
    final_candidates["Dataset name"]
    .apply(normalize_dataset_name)
)

final_candidates["identity_key"] = (
    final_candidates["current_paper_key"]
    + "||"
    + final_candidates["current_dataset_key"]
)


# Prepare exact-match information from the old 134-candidate audit
old_exact_lookup = old_audit.copy()

old_exact_lookup["audit_paper_key"] = (
    old_exact_lookup["Paper Link (URL)"]
    .apply(normalize_url)
)

old_exact_lookup["audit_dataset_key"] = (
    old_exact_lookup["Dataset name"]
    .apply(normalize_dataset_name)
)

old_exact_lookup["identity_key"] = (
    old_exact_lookup["audit_paper_key"]
    + "||"
    + old_exact_lookup["audit_dataset_key"]
)


prior_audit_cols = [
    "identity_key",
    "Country_split verified?",
    "Per country counts as tuples",
    "Minimum country count",
    "Split unit",
    "Same task and label format? ",
    "Any confounders e.g. level, topic etc",
    "Verdict? Pass or fail",
    "Evidence source",
    "Verifiable? If no, notes",
]

prior_audit_info = (
    old_exact_lookup[prior_audit_cols]
    .drop_duplicates("identity_key")
    .copy()
)

prior_audit_info = prior_audit_info.rename(columns={
    "Country_split verified?":
        "prior_country_split_verified",

    "Per country counts as tuples":
        "prior_per_country_counts",

    "Minimum country count":
        "prior_minimum_country_n",

    "Split unit":
        "prior_split_unit",

    "Same task and label format? ":
        "prior_same_task_and_label_format",

    "Any confounders e.g. level, topic etc":
        "prior_confounders",

    "Verdict? Pass or fail":
        "prior_verdict",

    "Evidence source":
        "prior_evidence_source",

    "Verifiable? If no, notes":
        "prior_verification_notes",
})


# Merge exact prior audit information
final_candidates = final_candidates.merge(
    prior_audit_info,
    on="identity_key",
    how="left",
    validate="many_to_one",
)


# Explicit flag
final_candidates["prior_exact_audit"] = np.where(
    final_candidates["prior_verdict"].notna(),
    "Yes",
    "No"
)


# ------------------------------------------------------------
# Put the useful audit flag near the front
# ------------------------------------------------------------

preferred_front = [
    "candidate_order",
    "Dataset name",
    "language",
    "candidate_type",
    "prior_exact_audit",
    "prior_verdict",
    "paper_url",
]

remaining_cols = [
    c for c in final_candidates.columns
    if c not in preferred_front
]

final_candidates = final_candidates[
    preferred_front + remaining_cols
]


# ------------------------------------------------------------
# Quick summary
# ------------------------------------------------------------

print("\nPRIOR AUDIT COVERAGE")
print("--------------------")
print(
    final_candidates["prior_exact_audit"]
    .value_counts()
    .to_string()
)

print("\nPrior verdicts among exact matches:")
print(
    final_candidates.loc[
        final_candidates["prior_exact_audit"].eq("Yes"),
        "prior_verdict"
    ]
    .value_counts(dropna=False)
    .to_string()
)


PRIOR AUDIT COVERAGE
--------------------
Yes    32
No     13

Prior verdicts among exact matches:
FAIL                    13
USABLE WITH CONTROLS    10
NOT VERIFIABLE           7
PASS                     2


In [33]:
final_candidates.to_csv(
    OUTPUT_FILE,
    index=False
)

print(f"\nSaved to: {OUTPUT_FILE.resolve()}")


Saved to: /home/jnwatu/Worldbench/Cross Country Funnel/language_not_enough_core_cross_country_candidates.csv
